# Module 08 — Tương quan Pearson & tương quan riêng phần
Notebook RIÊNG cho Module 08 — tái lập Pearson r + Spearman rho (dữ liệu lớp học) VÀ tương quan riêng phần (dữ liệu PISA thật) — phần riêng phần PSPP 2.1.1 KHÔNG hỗ trợ (`PARTIAL CORR is not yet implemented`), chỉ chạy được ở đây.

**Notebook này được viết để tự giải thích được, không cần mở kèm trang web** — mỗi phương pháp đều có phần "phương pháp này trả lời câu hỏi gì" trước khi chạy code, và phần "kết quả có ý nghĩa gì" sau khi chạy xong.

## 0. Tương quan Pearson dùng để trả lời câu hỏi gì?
Câu hỏi mà hệ số tương quan Pearson (r) trả lời: **"Hai biến số có xu hướng cùng tăng/giảm với nhau theo một đường THẲNG hay không, và mạnh tới mức nào?"** Không hơn, không kém. Cụ thể:

- **Dấu (+/−)**: cùng tăng-giảm (dương) hay ngược chiều (âm).
- **Độ lớn |r|** (0 tới 1): đường thẳng khớp với đám mây điểm dữ liệu tốt tới đâu — không phải "biến A ảnh hưởng biến B mạnh tới đâu".
- **r KHÔNG nói gì về nhân quả** — chỉ nói về việc CÙNG XUẤT HIỆN theo một khuôn mẫu tuyến tính. r cao có thể do A gây B, B gây A, một biến C gây cả hai, hoặc thuần tuý ngẫu nhiên (mẫu nhỏ).
- **Tính chất quan trọng nhất cho phần dưới của notebook này: r là ĐỐI XỨNG** — r(học_giờ, điểm_Toán) = r(điểm_Toán, học_giờ), tính theo chiều nào cũng ra đúng một con số. Đây là điều dễ gây hiểu lầm khi chuyển sang nói về "tăng X thì Y tăng bao nhiêu" — xem mục 5 để hiểu vì sao.

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' — CÙNG một seed=2026 dùng
    xuyên suốt các module 03-08, khớp chính xác với lop_hoc_240.csv/.sav
    trong repo (không phải dữ liệu thật, nhưng tái lập được 100%)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Đã tạo df:", df.shape)

## 1. Làm sạch outlier `study_hours` (giống Module 04/05)

In [ ]:
mask = df["study_hours"] > 20
df.loc[mask, "study_hours"] = df.loc[mask, "study_hours"] / 10

## 2. Pearson r: giờ tự học vs điểm Toán

In [ ]:
r, p = stats.pearsonr(df["study_hours"], df["math"])
print(f"Pearson r={r:.3f}, r²={r**2:.3f}, p={p:.2e}")

#### 📤 Đầu ra thật
r≈0,582, r²≈0,338 (giải thích ≈34% biến thiên điểm Toán), p<0,001. ✅ Khớp đúng bài.

### ❓ Câu hỏi hay gặp: "Tăng 1 giờ tự học thì điểm Toán tăng bao nhiêu — và NGƯỢC LẠI, tăng 1 điểm Toán thì giờ tự học tăng bấy nhiêu?"
**KHÔNG.** r chỉ cho biết ĐỘ MẠNH của quan hệ tuyến tính, không cho biết "tăng 1 đơn vị X thì Y tăng bao nhiêu" — đó là câu hỏi của HỆ SỐ HỒI QUY (slope, ký hiệu b), một đại lượng KHÁC hẳn và **không đối xứng**. Công thức: b(Y theo X) = r × (SD của Y / SD của X). Vì tỉ lệ SD(Y)/SD(X) khác với SD(X)/SD(Y) (trừ khi hai biến có cùng độ lệch chuẩn), hai chiều hồi quy cho ra hai con số KHÁC NHAU — xem minh chứng bằng số thật ngay dưới đây.

In [ ]:
sx = df["study_hours"].std(ddof=1); sy = df["math"].std(ddof=1)
b_math_theo_giohoc = r * sy / sx   # tăng 1 giờ tự học -> điểm Toán tăng bao nhiêu
b_giohoc_theo_math = r * sx / sy   # tăng 1 điểm Toán -> giờ tự học tăng bao nhiêu
print(f"SD(giờ tự học)={sx:.3f}  SD(điểm Toán)={sy:.3f}")
print(f"Tăng 1 giờ tự học/tuần -> điểm Toán tăng trung bình {b_math_theo_giohoc:.2f} điểm")
print(f"Tăng 1 điểm Toán -> giờ tự học/tuần tăng trung bình {b_giohoc_theo_math:.3f} giờ (~{b_giohoc_theo_math*60:.0f} phút)")

#### 📤 Đầu ra thật — đây chính là câu trả lời cho câu hỏi "2 chiều ra sao?"
Tăng 1 giờ tự học/tuần → điểm Toán tăng trung bình **≈2,51 điểm**. Nhưng tăng 1 điểm Toán → giờ tự học chỉ tăng trung bình **≈0,135 giờ (~8 phút)**. Hai con số này KHÔNG PHẢI nghịch đảo của nhau, và chênh lệch rất lớn (2,51 vs 0,135) — vì đơn vị đo của hai biến khác hẳn nhau (giờ vs điểm) VÀ độ phân tán (SD) của chúng khác nhau. r=0,582 là con số DUY NHẤT không đổi theo chiều; còn "tăng X thì Y tăng bao nhiêu" luôn phải hỏi rõ ĐANG QUY VỀ BIẾN NÀO làm biến phụ thuộc.

## 3. Spearman rho — đối chiếu với Pearson

In [ ]:
rho, p_rho = stats.spearmanr(df["study_hours"], df["math"])
print(f"Spearman rho={rho:.3f}, p={p_rho:.2e}")

#### 📤 Đầu ra thật
rho≈0,554 — khá gần Pearson r vì quan hệ khá tuyến tính. **Phương pháp này dùng để làm gì khác Pearson?** Spearman rho trả lời CÙNG câu hỏi (2 biến có xu hướng cùng tăng/giảm không) nhưng dựa trên THỨ HẠNG thay vì giá trị thô — vì vậy không giả định quan hệ phải là đường thẳng hoàn hảo, chỉ cần "đơn điệu cùng chiều" (biến này tăng thì biến kia CÓ XU HƯỚNG tăng, không nhất thiết theo tỉ lệ cố định). Dùng khi nghi ngờ có ngoại lai mạnh hoặc quan hệ cong nhưng vẫn đơn điệu.

In [ ]:
fig, ax = plt.subplots()
sns.regplot(data=df, x="study_hours", y="math", ax=ax, scatter_kws={"alpha":.5})
ax.set_title("Giờ tự học/tuần vs điểm Toán (n=240, sau khi xử lý 1 ngoại lai)")
plt.show()

## 4. Tương quan riêng phần — dữ liệu PISA thật (195 trường)
**Phương pháp này trả lời câu hỏi gì?** Tương quan riêng phần trả lời: "Nếu ta 'khoá cố định' ảnh hưởng của một biến thứ ba Z, quan hệ THẬT giữa X và Y còn lại bao nhiêu?" — dùng khi nghi ngờ một phần (hoặc toàn bộ) tương quan quan sát được giữa X và Y thực ra là do cả hai cùng liên quan tới Z, chứ không phải X và Y trực tiếp liên quan nhau.

Tải dữ liệu cấp trường trực tiếp từ GitHub — biến `EDUSHORT` (thiếu học liệu), `sci_mean` (điểm Khoa học TB trường), `STAFFSHORT` (thiếu nhân sự, biến nghi ngờ gây nhiễu).

In [ ]:
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
d = truong[["EDUSHORT", "sci_mean", "STAFFSHORT"]].dropna()
print("n =", len(d))

r_xy = stats.pearsonr(d["EDUSHORT"], d["sci_mean"])[0]
r_xz = stats.pearsonr(d["EDUSHORT"], d["STAFFSHORT"])[0]
r_yz = stats.pearsonr(d["sci_mean"], d["STAFFSHORT"])[0]
r_partial = (r_xy - r_xz*r_yz) / np.sqrt((1-r_xz**2)*(1-r_yz**2))
print(f"r(EDUSHORT, điểm KH) gốc = {r_xy:.3f}")
print(f"r(EDUSHORT, STAFFSHORT)  = {r_xz:.3f}")
print(f"r(điểm KH, STAFFSHORT)   = {r_yz:.3f}")
print(f"Tương quan riêng phần    = {r_partial:.3f}")

#### 📤 Đầu ra thật
Tương quan gốc EDUSHORT × điểm Khoa học: r=-0,039 (rất yếu). r(EDUSHORT, STAFFSHORT)≈0,61 (khá cao — biến nghi ngờ gây nhiễu). r(điểm KH, STAFFSHORT)≈0,002 (gần như 0). Tương quan riêng phần sau khi kiểm soát STAFFSHORT: r=-0,051 — **gần như không đổi**.

**Vì sao gần như không đổi?** Công thức tương quan riêng phần chỉ thay đổi mạnh khi biến kiểm soát Z tương quan THẬT với CẢ HAI biến chính. Ở đây, thiếu nhân sự gần như không liên quan điểm Khoa học (r≈0,002) — nên "loại bỏ" nó không thay đổi nhiều bức tranh ban đầu. Đây là kết quả thật, không phải lúc nào dữ liệu cũng cho ra ví dụ kịch tính như trong sách.

## 5. Minh chứng đầy đủ: tính KHÔNG đối xứng của hồi quy — EDULEAD ↔ NEGSCLIM
Đây là ví dụ trực tiếp trả lời câu hỏi: *"Nếu Educational Leadership tăng 1 điểm làm Negative School Climate giảm 0,06 điểm, thì Negative School Climate tăng 1 điểm có làm Educational Leadership giảm một số RẤT LỚN (kiểu 'mười mấy điểm') không?"*

**Trả lời ngắn gọn: KHÔNG — nhưng hai chiều vẫn KHÁC NHAU, chỉ là khác theo một tỉ lệ có công thức rõ ràng, không phải ngẫu nhiên hay "khuếch đại kiểu nghịch đảo".**

In [ ]:
d2 = truong[["EDULEAD", "NEGSCLIM"]].dropna()
r2, p2 = stats.pearsonr(d2["EDULEAD"], d2["NEGSCLIM"])
sx2 = d2["EDULEAD"].std(ddof=1); sy2 = d2["NEGSCLIM"].std(ddof=1)
b_neg_theo_edu = r2 * sy2 / sx2   # EDULEAD tăng 1 -> NEGSCLIM đổi bao nhiêu
b_edu_theo_neg = r2 * sx2 / sy2   # NEGSCLIM tăng 1 -> EDULEAD đổi bao nhiêu
print(f"n={len(d2)}, r={r2:.4f}, p={p2:.4f}  (p>0,05 -> KHÔNG có ý nghĩa thống kê, xem cảnh báo bên dưới)")
print(f"SD(EDULEAD)={sx2:.4f}  SD(NEGSCLIM)={sy2:.4f}")
print(f"EDULEAD tăng 1 điểm  -> NEGSCLIM đổi trung bình {b_neg_theo_edu:+.4f} điểm")
print(f"NEGSCLIM tăng 1 điểm -> EDULEAD đổi trung bình  {b_edu_theo_neg:+.4f} điểm")
print(f"Tỉ lệ 2 slope = {b_neg_theo_edu/b_edu_theo_neg:.3f}  =  (SD_NEGSCLIM / SD_EDULEAD)² = {(sy2/sx2)**2:.3f}")

#### 📤 Đầu ra thật — và cách đọc đúng
r=-0,057, **p=0,427 → KHÔNG có ý nghĩa thống kê** (quan hệ này yếu tới mức có thể chỉ do may rủi lấy mẫu — nêu ra đây THUẦN TUÝ để minh hoạ toán học của tính bất đối xứng, KHÔNG phải để kết luận "lãnh đạo tốt làm giảm môi trường tiêu cực" — với p này, không đủ bằng chứng nói vậy).

Về mặt số học của câu hỏi "2 chiều ra sao": EDULEAD tăng 1 điểm → NEGSCLIM đổi trung bình **-0,085 điểm**. NEGSCLIM tăng 1 điểm → EDULEAD đổi trung bình **-0,038 điểm**. Hai số này khác nhau **đúng 2,21 lần** — và 2,21 chính xác bằng (SD của NEGSCLIM / SD của EDULEAD)² = (1,336/0,899)². **Không phải "mười mấy lần"** như trực giác đôi khi đoán — độ chênh giữa 2 chiều hồi quy luôn bị giới hạn bởi TỈ LỆ ĐỘ LỆCH CHUẨN bình phương của 2 biến, không phải một con số tuỳ tiện.

**Quy tắc tổng quát cần nhớ:**
1. Hệ số tương quan r: **1 con số duy nhất**, đối xứng, không phụ thuộc chiều.
2. Hệ số hồi quy b (slope): **2 con số khác nhau** tuỳ theo biến nào là phụ thuộc — công thức b(Y theo X) = r×SD(Y)/SD(X).
3. Tích của 2 slope luôn bằng r²: b(Y theo X) × b(X theo Y) = r² — đây là công thức kiểm tra nhanh 2 slope có tính đúng không.
4. Hai slope chỉ BẰNG NHAU khi SD(X) = SD(Y) — hiếm khi xảy ra với 2 biến đo bằng thang khác nhau.

In [ ]:
# Kiểm chứng công thức tích 2 slope = r^2
print(f"b_yx * b_xy = {b_neg_theo_edu * b_edu_theo_neg:.5f}")
print(f"r^2         = {r2**2:.5f}")
print("Hai số này BẰNG NHAU -> xác nhận công thức đúng, không phải trùng hợp.")